In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt


In [2]:
# --- PHẦN 1: CLASS DECISION TREE (TỰ CODE TỪ GĐ 3) ---

class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx  
        self.threshold = threshold      
        self.left = left                
        self.right = right              
        self.value = value              

class MyDecisionTreeRegressor:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _calculate_variance_reduction(self, y, y_left, y_right):
        var_total = np.var(y)
        weight_left = len(y_left) / len(y)
        weight_right = len(y_right) / len(y)
        reduction = var_total - (weight_left * np.var(y_left) + weight_right * np.var(y_right))
        return reduction

    def _get_best_split(self, X, y):
        best_split = {}
        best_var_reduction = -float("inf")
        n_samples, n_features = X.shape

        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_idx = X[:, feature_idx] <= threshold
                right_idx = X[:, feature_idx] > threshold

                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue

                var_reduction = self._calculate_variance_reduction(y, y[left_idx], y[right_idx])
                if var_reduction > best_var_reduction:
                    best_var_reduction = var_reduction
                    best_split = {
                        "feature_idx": feature_idx,
                        "threshold": threshold,
                        "left_idx": left_idx,
                        "right_idx": right_idx
                    }
        return best_split

    def _build_tree(self, X, y, depth=0):
        n_samples = X.shape[0]
        if (n_samples < self.min_samples_split) or (depth >= self.max_depth):
            return Node(value=np.mean(y))

        split = self._get_best_split(X, y)
        if not split:
            return Node(value=np.mean(y))

        left = self._build_tree(X[split["left_idx"]], y[split["left_idx"]], depth + 1)
        right = self._build_tree(X[split["right_idx"]], y[split["right_idx"]], depth + 1)
        return Node(feature_idx=split["feature_idx"], threshold=split["threshold"], left=left, right=right)

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])


In [3]:
# --- PHẦN 2: RANDOM FOREST SỬ DỤNG CÂY TỰ CODE ---

class MyRandomForestRegressor:
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []

    def _bootstrap(self, X, y):
        n_samples = X.shape[0]
        idx = np.random.choice(n_samples, n_samples, replace=True)
        return X[idx], y[idx]

    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.trees = []
        for _ in range(self.n_estimators):
            X_s, y_s = self._bootstrap(X, y)
            tree = MyDecisionTreeRegressor(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(X_s, y_s)
            self.trees.append(tree)

    def predict(self, X):
        preds = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(preds, axis=0)


In [4]:
# --- PHẦN 3: HUẤN LUYỆN VÀ ĐÁNH GIÁ ---

# TODO: Load dữ liệu tại đây
# X_train, X_test, y_train, y_test = ... 

model = MyRandomForestRegressor(n_estimators=10, max_depth=7)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = np.mean((y_test - y_pred)**2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_pred))
ss_res = np.sum((y_test - y_pred)**2)
ss_tot = np.sum((y_test - np.mean(y_test))**2)
r2 = 1 - (ss_res / ss_tot)

print(f'--- Kết quả đánh giá ---')
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R2:   {r2:.4f}')

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--')
plt.xlabel('Giá trị thực tế')
plt.ylabel('Giá trị dự đoán')
plt.title('So sánh Thực tế vs Dự đoán (My Random Forest)')
plt.show()


NameError: name 'X_train' is not defined